# CC3092 – Deep Learning · Laboratorio 8
**Mini-GPT con muestreo controlado + embeddings contextuales (BERT) vs. estáticos (Word2Vec)**

Todo el Mini-GPT está construido *from scratch* con tensores y `nn.Parameter`/proyecciones lineales:
- Sin `nn.MultiheadAttention`, `nn.Transformer`, `nn.LayerNorm`.
- LayerNorm, softmax, cross-entropy, Positional Encoding (seno/coseno), atención multi-cabeza, PCA y similitud coseno programados a mano.
- Minibatches generados manualmente (índices), sin `DataLoader`.
- Learning rate con decaimiento expuesto en el bucle de entrenamiento + `clip_grad_norm_`.

Dependencias: `pip install torch numpy matplotlib pandas transformers gensim`

In [4]:
import os, math, time, re, itertools, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)
print("device:", device)

device: cpu


## Task 1.1 – Dataset, vocabulario y batches

In [5]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "input.txt"
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(URL, DATA_PATH)
with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()
print("Caracteres totales:", len(text))

# Vocabulario de caracteres únicos y mappings bidireccionales char <-> índice
chars = sorted(set(text))
vocab_size = len(chars)
assert vocab_size == 65, f"Se esperaban 65 caracteres, hay {vocab_size}"
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

# Texto completo como secuencia de índices + split temporal 90/10 (sin barajar)
data = torch.tensor(encode(text), dtype=torch.long)
n_split = int(0.9 * len(data))
train_data, val_data = data[:n_split], data[n_split:]
print("train:", len(train_data), "| val:", len(val_data), "| vocab:", vocab_size)

# ---------------- Hiperparámetros ----------------
block_size = 128
d_model    = 128
n_heads    = 4
n_layers   = 3
batch_size = 64
lr         = 3e-4
epochs     = 10
# Extras (no contradicen el enunciado)
dropout      = 0.1
d_ff         = 4 * d_model
grad_clip    = 1.0      # clip_grad_norm_ contra explosión de gradientes
warmup_steps = 100
lr_min       = 3e-5
STRIDE       = 64       # paso de la ventana deslizante en entrenamiento (menor = más pasos por época)

Caracteres totales: 1115394
train: 1003854 | val: 111540 | vocab: 65


In [6]:
# ---------------- Generación manual de batches (ventanas deslizantes por índices) ----------------
def make_window_starts(n_tokens, block_size, stride):
    # Un inicio s es válido si s + block_size + 1 <= n_tokens (necesitamos el target desplazado +1)
    last_start = n_tokens - block_size - 1
    return torch.arange(0, last_start + 1, stride)

def build_batch(split_data, starts, block_size):
    # idx[b, t] = starts[b] + t  ->  x = data[idx],  y = data[idx + 1]  (target = input desplazado 1 paso)
    offs = torch.arange(block_size)
    idx = starts[:, None] + offs[None, :]
    x = split_data[idx]
    y = split_data[idx + 1]
    return x.to(device), y.to(device)

def epoch_batches(split_data, block_size, batch_size, stride, shuffle=True, drop_last=True):
    starts = make_window_starts(len(split_data), block_size, stride)
    if shuffle:
        starts = starts[torch.randperm(len(starts))]    # permutación de índices
    n = len(starts)
    end = (n // batch_size) * batch_size if drop_last else n
    for i in range(0, end, batch_size):
        yield build_batch(split_data, starts[i:i + batch_size], block_size)

# Sanity check: y es x desplazado un paso
xb, yb = next(epoch_batches(train_data, block_size, batch_size, STRIDE))
assert torch.equal(xb[:, 1:], yb[:, :-1])
print("batch:", xb.shape, yb.shape)
print("x[0][:30] ->", repr(decode(xb[0, :30].cpu())))
print("y[0][:30] ->", repr(decode(yb[0, :30].cpu())))

batch: torch.Size([64, 128]) torch.Size([64, 128])
x[0][:30] -> 'e,\nA thirsty evil; and when we'
y[0][:30] -> ',\nA thirsty evil; and when we '


### Mini-GPT desde cero
Bloques: Positional Encoding sinusoidal, LayerNorm manual (varianza sesgada), atención causal multi-cabeza explícita, FFN con GELU, arquitectura Pre-LN.

In [7]:
# ---------------- Softmax estable (manual) ----------------
def softmax(z, dim=-1):
    # softmax(z)_i = exp(z_i - max z) / sum_j exp(z_j - max z)
    z = z - z.max(dim=dim, keepdim=True).values
    e = torch.exp(z)
    return e / e.sum(dim=dim, keepdim=True)

# ---------------- Cross-entropy manual ----------------
def cross_entropy(logits, targets, reduction="mean"):
    # L_t = -log softmax(z)_{y_t} = logsumexp(z) - z_{y_t}
    V = logits.size(-1)
    logits = logits.reshape(-1, V)
    t = targets.reshape(-1)
    m = logits.max(dim=-1, keepdim=True).values
    lse = m.squeeze(-1) + torch.log(torch.exp(logits - m).sum(dim=-1))
    nll = lse - logits.gather(1, t[:, None]).squeeze(1)
    return nll.sum() if reduction == "sum" else nll.mean()

# ---------------- LayerNorm manual ----------------
class LayerNormManual(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d))
        self.beta = nn.Parameter(torch.zeros(d))
    def forward(self, x):
        mu = x.mean(dim=-1, keepdim=True)                       # mu = (1/d) sum x_i
        var = ((x - mu) ** 2).mean(dim=-1, keepdim=True)        # sigma^2 sesgada = (1/d) sum (x_i - mu)^2
        x_hat = (x - mu) / torch.sqrt(var + self.eps)           # x_hat = (x - mu) / sqrt(sigma^2 + eps)
        return self.gamma * x_hat + self.beta                   # y = gamma * x_hat + beta

# ---------------- Positional Encoding sinusoidal ----------------
def sinusoidal_positional_encoding(max_len, d_model):
    # PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    # PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)          # (T,1)
    two_i = torch.arange(0, d_model, 2, dtype=torch.float32)               # 2i
    div = torch.exp(-math.log(10000.0) * two_i / d_model)                  # 1 / 10000^(2i/d)
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

# ---------------- Atención causal multi-cabeza ----------------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.h = n_heads
        self.dk = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.p = dropout
        self.register_buffer("causal", torch.tril(torch.ones(block_size, block_size, dtype=torch.bool)))
    def forward(self, x):
        B, T, C = x.shape
        # Q = X W_q, K = X W_k, V = X W_v  -> separar en cabezas (B, h, T, dk)
        q = self.W_q(x).view(B, T, self.h, self.dk).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.h, self.dk).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.h, self.dk).transpose(1, 2)
        # scores = Q K^T / sqrt(d_k)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)
        # máscara causal: posición t no atiende a j > t
        scores = scores.masked_fill(~self.causal[:T, :T], float("-inf"))
        A = softmax(scores, dim=-1)                       # A = softmax(QK^T / sqrt(dk))
        A = F.dropout(A, p=self.p, training=self.training)
        heads = A @ v                                      # head_i = A_i V_i
        concat = heads.transpose(1, 2).contiguous().view(B, T, C)   # Concat(head_1..head_h)
        return self.W_o(concat)                            # MHA = Concat(...) W_o

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.W1 = nn.Linear(d_model, d_ff)
        self.W2 = nn.Linear(d_ff, d_model)
        self.p = dropout
    def forward(self, x):
        # FFN(x) = W2 GELU(W1 x + b1) + b2
        return F.dropout(self.W2(F.gelu(self.W1(x))), p=self.p, training=self.training)

class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size, dropout):
        super().__init__()
        self.ln1 = LayerNormManual(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = LayerNormManual(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.p = dropout
    def forward(self, x):
        x = x + F.dropout(self.attn(self.ln1(x)), p=self.p, training=self.training)   # x + MHA(LN(x))
        x = x + self.ffn(self.ln2(x))                                                  # x + FFN(LN(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, block_size, dropout):
        super().__init__()
        self.d_model = d_model
        self.block_size = block_size
        self.p = dropout
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pe", sinusoidal_positional_encoding(block_size, d_model))
        self.blocks = nn.ModuleList([Block(d_model, n_heads, d_ff, block_size, dropout) for _ in range(n_layers)])
        self.ln_f = LayerNormManual(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) * math.sqrt(self.d_model) + self.pe[:T]   # emb * sqrt(d) + PE
        x = F.dropout(x, p=self.p, training=self.training)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))                                   # logits (B, T, V)

model = MiniGPT(vocab_size, d_model, n_heads, n_layers, d_ff, block_size, dropout).to(device)
print("Parámetros:", sum(p.numel() for p in model.parameters()))

Parámetros: 610625
